# Targets: run an experiment here and on the cluster

An experiment is a spec in `config/experiments/` (which runs to train). *Where* it runs is a
**target**: `local` is this PC, `taltech` is the cluster. Both are used the same way:

| Call | What it does |
|---|---|
| `target.start(spec, profile)` | queue the experiment and start it; returns at once with its name |
| `target.runs(name)` | how each run is doing (a `RunState` per run) |
| `target.stop(name)` | stop the experiment (or one run: `stop(name, run)`) |
| `target.sync(name)` | make the results available on this PC |

This notebook walks through them with the smallest experiment, `config/experiments/smoke.json`
(two tiny PPO runs), and always with `profile="test"`: about one training iteration per
run, a few minutes in total. The test profile is an experiment of its own, named
`smoke--test`.

**How the cluster is reached.** From this PC, `targets.get("taltech")` gives a
`RemoteTarget`. It keeps one SSH connection to the cluster's login node (the alias
`taltech` in `~/.ssh/config`) and runs there the same command you would type on the
cluster, `python -m core.runner ...`, which submits the runs to SLURM. Training happens
on the compute nodes; this notebook only starts, watches and fetches, and closing it
does not stop the runs. You need the university VPN for the connection.

In [ ]:
import dataclasses
import sys
import time
from pathlib import Path

import pandas as pd

# Repository root: found from wherever the notebook runs, and put on the path.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "core" / "rl").is_dir())
sys.path.insert(0, str(ROOT))

from core.runner import targets

SPEC = ROOT / "config" / "experiments" / "smoke.json"  # a path, or "config/experiments/smoke.json" from ROOT

Two helpers, used for both targets. `runs_table` turns the runs into a table;
`watch` prints the runs every few seconds until none is queued or running. Stopping
the `watch` cell (interrupt) only stops watching: the runs go on.

In [ ]:
BUSY = ("queued", "running")


def runs_table(runs) -> pd.DataFrame:
    """One row per run: state, progress, scores (a `RunState` each)."""
    table = pd.DataFrame([dataclasses.asdict(run) for run in runs])
    return table.set_index("name") if len(table) else table


def watch(target, name, every_s):
    """Print the runs of `name` every `every_s` seconds until none is busy."""
    while True:
        runs = target.runs(name)
        print(time.strftime("%H:%M:%S"), name)
        for run in runs:
            steps = f"{run.env_steps}/{run.budget or '?'}"
            print(
                f"  {run.name:<18} {run.state:<10} slurm={run.slurm_state or '-':<10} "
                f"host={run.host or '-':<10} steps={steps}"
            )
        if runs and not any(run.state in BUSY for run in runs):
            return runs
        time.sleep(every_s)

## 1. Local test

`local` starts a detached worker on this PC: it outlives the notebook. It writes
`runs/smoke--test/` (the spec, one folder per run with its `status.json`) and trains
the runs as separate processes, up to `max_parallel` at a time.

In [ ]:
local = targets.get("local")
name = local.start(SPEC, profile="test")
name

Each line is one run: its state (`queued` -> `running` -> `completed`, or `failed`),
the SLURM state (empty on this PC), `host` (the machine that runs it) and the
environment steps done of its budget. This takes a few minutes: Ray starts for each run.

In [ ]:
watch(local, name, every_s=5)

The runs as a table. `last_eval` is the team return of the trained policy in the last
evaluation during training, `heldout` its team return on the held-out test days
(`heldout.csv`), `error` what went wrong in a failed run.

In [ ]:
runs_table(local.runs(name))

## 2. The cluster

`check()` asks the cluster whether it is ready, and compares its code with yours. The
cluster runs **its own checkout** of the repository, not the files on this PC, so an
experiment can only start there when both have the same code. The answer is one report:

- `problems`: what stops a start, each with what to do (the cluster cannot be reached,
  usually the VPN; its own check failed; this PC and the cluster are at different
  commits; the cluster has uncommitted changes);
- `ok`: true when there are no problems;
- `local_commit`, `cluster_commit`: the two commits;
- `active`: experiments with queued or running runs on the cluster. A queued task runs
  the cluster's checkout as it is when it starts, so do not update or edit it meanwhile.

If there are problems, commit here and run `git shipc "message"` in a terminal: it
pushes and updates the cluster. With a spec, `check(SPEC)` also reports a spec or
scenario that is not committed. The first call opens the connection and takes a few
seconds.

In [ ]:
taltech = targets.get("taltech")
taltech.check(SPEC)

`start` runs `check(SPEC)` itself and refuses with its problems, so it never submits
code that is not the cluster's. Otherwise the cluster submits one SLURM array task per
run and returns at once.

In [ ]:
name = taltech.start(SPEC, profile="test")
name

Watching the cluster, every 20 seconds. Next to the run's state you see SLURM's state
of its task (`PENDING` while it waits for a node, `RUNNING`, `COMPLETED`), the node it
runs on (`host`) and the steps done of its budget. Each look is one command on the
cluster, so there is no need to look more often.

In [ ]:
watch(taltech, name, every_s=20)

The results are files on the cluster. `sync` copies them to `runs/@taltech/<name>/` on
this PC as a fresh copy that replaces the previous one (also while the runs are going;
if the copy fails the previous one stays). The large checkpoints and SLURM's own logs
stay on the cluster. `taltech.copied` is the number of files the last `sync` copied.

In [ ]:
folder = taltech.sync(name)
print(folder, "-", taltech.copied, "files copied")

`local_runs` reads that copy like a local experiment (no connection needed), so this
table is what you analyse. `heldout` is the trained policy's team return on the test days.

In [ ]:
runs_table(taltech.local_runs(name))

## 3. Stopping

`stop` cancels the experiment's SLURM tasks and marks the runs that were queued or
running as `stopped`. Uncomment to try it on a running experiment; the name of a run is
a row of the tables above.

In [ ]:
# taltech.stop(name)                 # the whole experiment
# taltech.stop(name, "ppo_dtde_s0")  # one run

## 4. Closing

This ends the SSH connection and the thread that holds it. Using `taltech` again
opens a new one. (If you forget, it ends with the kernel.)

The same steps from a terminal, through the command line, with `--json` for a program
to read:

```
python -m core.runner check --target taltech
python -m core.runner start config/experiments/smoke.json --profile test --target taltech
python -m core.runner status smoke--test --target taltech
python -m core.runner sync smoke--test --target taltech
python -m core.runner stop smoke--test --target taltech
```

In [ ]:
taltech.remote.close()